In [1]:
import pandas as pd
from pandasql import sqldf

In [2]:
# read cleaned up files using pandas 
by_state = pd.read_csv("Earnings_By_State.csv")
by_county = pd.read_csv("Earnings_By_County.csv")

In [3]:
# rename the column back to state 
by_state = by_state.rename(columns = {"Unnamed: 0": "State"})

In [4]:
# Average total income in U.S. 
avg_income = sqldf("""SELECT 
                            AVG(total) AS Total, 
                            AVG(`Less than high school graduate`) AS `Less than high school graduate`, 
                            AVG(`High school graduate`) AS `High school graduate`, 
                            AVG(`Some college or associate's degree`) AS `Some college or associate's degree`,
                            AVG(`Bachelor's degree`) AS `Bachelor's degree`,
                            AVG(`Graduate or professional degree`) AS `Graduate or professional degree`
                      FROM by_state""")
avg_income = avg_income.round(0).astype(int) # round to whole numbers 

In [5]:
# State with highest average total income 
highest_avg_state = sqldf("SELECT * FROM by_state ORDER BY total DESC LIMIT 1")

# State with lowest average total income 
lowest_avg_state = sqldf("SELECT * FROM by_state ORDER BY total LIMIT 1")

# California average income 
CA_avg_state = sqldf("SELECT * FROM by_state WHERE state = 'California' ORDER BY total")

In [6]:
# State with highest average income for each education level 

# Less than high school graduate education level
lessHS_avg_state = sqldf("SELECT * FROM by_state ORDER BY `less than high school graduate` DESC LIMIT 1")
# High school graduate education level 
HS_avg_state = sqldf("SELECT * FROM by_state ORDER BY `high school graduate` DESC LIMIT 1")
# Some college or Associate's degree education level 
somedegree_avg_state = sqldf("SELECT * FROM by_state ORDER BY `some college or associate's degree` DESC LIMIT 1")
# Bachelor's degree education level 
bachdegree_avg_state = sqldf("SELECT * FROM by_state ORDER BY `bachelor's degree` DESC LIMIT 1")
# Above Bachelor's degree education level 
abovebach_avg_state = sqldf("SELECT * FROM by_state ORDER BY `graduate or professional degree` DESC LIMIT 1")

# State with lowest average income level for each education level 

# Less than high school graduate education level
lessHS_avg_state = sqldf("SELECT * FROM by_state ORDER BY `less than high school graduate` LIMIT 1")
# High school graduate education level 
HS_avg_state = sqldf("SELECT * FROM by_state ORDER BY `high school graduate` LIMIT 1")
# Some college or Associate's degree education level 
somedegree_avg_state = sqldf("SELECT * FROM by_state ORDER BY `some college or associate's degree` LIMIT 1")
# Bachelor's degree education level 
bachdegree_avg_state = sqldf("SELECT * FROM by_state ORDER BY `bachelor's degree` LIMIT 1")
# Above Bachelor's degree education level 
abovebach_avg_state = sqldf("SELECT * FROM by_state ORDER BY `graduate or professional degree` LIMIT 1")

In [7]:
# Split the name into both county and state so they can be grouped later
by_county[["County", "State"]] = by_county.NAME.str.split(",", expand = True)

In [8]:
# County with highest average income 
highest_avg_county = sqldf("SELECT * FROM by_county ORDER BY total DESC LIMIT 1")

# County with lowest average income 
lowest_avg_county = sqldf("SELECT * FROM by_county ORDER BY total LIMIT 1")

# Only California counties 
CA_avg_county = sqldf("SELECT * FROM by_county WHERE state = ' California' ORDER BY total DESC")

In [9]:
# County with highest average income per state 
high_county_per = sqldf("""SELECT 
                               County, 
                               C.State, 
                               Name, 
                               Total, 
                               `Less than high school graduate`, 
                               `High school graduate`, 
                               `Some college or associate's degree`,
                               `Bachelor's degree`,
                               `Graduate or professional degree`
                           FROM
                               (SELECT State, MAX(total) AS highest FROM by_county GROUP BY state) AS high 
                           JOIN 
                               by_county AS C
                           ON 
                               high.state = C.state AND high.highest = C.total""")

In [10]:
# County with lowest average income per state 
low_county_per = sqldf("""SELECT
                              County, 
                              C.State, 
                              Name, 
                              Total, 
                              `Less than high school graduate`, 
                              `High school graduate`, 
                              `Some college or associate's degree`,
                              `Bachelor's degree`,
                              `Graduate or professional degree`
                          FROM
                              (SELECT State, MIN(total) AS lowest FROM by_county GROUP BY state) AS low 
                          JOIN 
                              by_county AS C
                          ON 
                              low.state = C.state AND low.lowest = C.total""")